# A first look at `pysignet`


This notebook shows the basic concepts of the `pysignet` library with a simple logical constraint and neural predicates.

By the end of this notebook, we will have seen:

1. How to define predicates, variables, and logical expressions
2. How to compile an expression into a differentiable computation graph using `compile_logic`
3. Why soft and boolean satisfaction differ, and when each is useful
4. How to use `logic_to_loss` to train a neural network to satisfy a constraint

A core concept in `pysignet` is the idea of a [named neuron](https://svivek.com/writing/2026-01-28-named-neurons.html). *A named neuron is a scalar activation in a neural network that admits a mapping to a symbol in a formal system, enabling logical reasoning over the network.*

In other words, these are specific neurons that we give symbolic names (like `P`, `Q`, `Digit`, etc) and use in logical expressions. These can be 
- Neural network outputs (i.e. final layer activations)
- Deterministic functions of inputs
- Intermediate activations


Importantly, named neurons represent predicates whose arguments are the inputs to the neural networks. So if an input is `X`, we can write `P(X)`, which would be mapped to a neuron.

This example uses only output named neurons. That is, it only uses final outputs of neural networks and functions that we can combine with logical operators. Other notebooks show more complex behaviors.

First, let us get some imports out of the way. The library is built on PyTorch, so we have the `torch` imports. We import `pysignet as psn`, which gives us everything we need: `Symbol`, `Variable`, and the logical operators `And`, `Or`, `Not`, `Implies`, `Equivalent`, `ForAll`, and `Exists`.

In [35]:
import torch
import torch.nn as nn 
import pysignet as psn

## A logical expression

To get started, let us create the predictes and variables to construct the following logical expression:
$$\forall X, P(X) \wedge \left(Q(X) \vee \neg R(X)\right)$$

Here, we have three predicates `P`, `Q` and `R`, all of which operate on the same argument `X`. The rule says that for any input `X`, the predicate `P` should hold and either the predicate `Q` should hold or the predicate `R` should *not*. 

First, let us create the predicates and the variable.

In [36]:
P, Q, R = psn.Symbol ("P Q R")
X = psn.Variable("X")


We can now use `pysignet`'s boolean operators `And`, `Or` and `Not` (re-exported from SymPy for convenience) to construct the expression above.

Now we encounter the first design choice in `pysignet`: Any variable that is not explicitly bound to a quantifier is assumed to be universally quantified. So we need not write the universal quantification explicitly when we construct the logical expression.

> **Power users**: `pysignet` re-exports the standard SymPy boolean operators `And`, `Or`, `Not`, `Implies`, and `Equivalent` so you don't need to `import sympy as sp` for basic use. If you need advanced SymPy features (simplification, CNF conversion, etc.), you can still `import sympy as sp`; the operators are the same objects.

In [37]:
expr = psn.And(P(X), psn.Or(Q(X), psn.Not(R(X))))

print(f"Logic expression: {expr}")

Logic expression: P(X) & (Q(X) | ~R(X))


## The neural networks

At this point, all we have is a purely symbolic expression. Next, let us create the "neural" parts of this neuro-symbolic synthesis. 

First, let us create a neural network that computes a soft version of the predicate `P`.  For this exercise, it is a simple two layer MLP.

In [38]:
model_p = nn.Sequential(nn.Linear(10, 5), 
                nn.ReLU(), 
                nn.Linear(5, 1),
                nn.Sigmoid())

The predicates `Q` and `R` can also be mapped to neural networks. But the library allows for more: they can be mapped to *any* computation graphs we construct using torch with or without learnable parameters. 

To illustrate that, we can create two deterministic functions of the inputs:

* The deterministic function `pred_q` whose output becomes the predicate `Q` for any given `x`
* Another deterministic function `pred_r` whose output becomes the predicate `R` for any given `x`

In [39]:
def pred_q(x: torch.Tensor) -> torch.Tensor:
    return (x.sum(dim=-1, keepdim=True) > 0).float()

def pred_r(x: torch.Tensor) -> torch.Tensor:
    return torch.sigmoid(x.mean(dim=-1, keepdim=True)).squeeze(-1)

## Binding the neural and the symbolic expressions

Now, we are ready to instantiate the named neuron abstraction. To do that, we need to map the symbolic predicates to their soft counterparts with a python `dict`. The mapping admits any functions, lambdas or `nn.Module`s. We have mapped the predicate `Q` to a lambda that takes one argument because the predicate takes one argument `X`.

In [40]:
predicates = {
    "P": model_p,
    "Q": lambda x: pred_q(x).squeeze(-1),
    "R": pred_r
}

## From logic to differentiable logic

At this point, we have all the pieces we need to convert the logic into differentiable units. `pysignet` provides a single convenience function `compile_logic` to compile logic to differentiable units. 

In [41]:
compiled = psn.compile_logic(expr, predicates)

What we have done is to construct a computation graph that is a soft version of the boolean expression, where the truth values of the predicates are computed by their named neurons. By default, the computation graph uses the R-product t-norm relaxation, but `compile_logic` allows other compilers.

We can use the compiled expression to check soft-satisfaction of the predicates for any data. For this example, let us use a random batch of data. Let us now create the data.

In [42]:
batch_size = 32
x = torch.randn(batch_size, 10)

We still need to tell the expression that the variable `X` in the logical expression maps to the `x` that we have constructed. The compiled logic provides a natural way to do this. First, we can ask for the boolean value of the logical expression, given the decisions about the predicates `P`, `Q` and `R`.

In [43]:
satisfaction = compiled(X=x, return_boolean=True)
satisfaction

tensor([False, False, False, False, False, False, False, False, False, False,
        False,  True, False,  True,  True,  True, False, False, False, False,
        False, False,  True,  True,  True, False, False, False, False, False,
        False, False])

But the more interesting thing happens when we remove the `return_boolean` flag to get a soft satisfaction. This gives us the result of a forward pass of the computation graph corresponding to our logical expression, with the variable `X` is bound to `x`, and the predicates bound to their respective named neurons.

In [44]:
soft_satisfaction = compiled(X=x)
soft_satisfaction

tensor([0.4323, 0.2622, 0.2630, 0.4921, 0.3988, 0.2102, 0.4821, 0.4366, 0.4860,
        0.4488, 0.4468, 0.3190, 0.2267, 0.5306, 0.5810, 0.2901, 0.2096, 0.2744,
        0.4731, 0.4960, 0.4561, 0.4619, 0.2617, 0.3465, 0.2834, 0.3713, 0.2183,
        0.3996, 0.2728, 0.2993, 0.4109, 0.3317], grad_fn=<ProdBackward1>)

### Technical note: Why soft and boolean satisfaction differ

You may notice that many `soft_satisfaction` values are below 0.5, yet the corresponding `satisfaction` (boolean) values are `True`. This is expected behavior! It arises due to the fundamental difference between the t-norm evaluation that populates the soft values, and boolean logic.

Consider `P(X) & Q(X)` where P=0.7 and Q=0.6:

| Method | Calculation | Result |
|--------|-------------|--------|
| **Soft (product t-norm)** | $0.7 \times 0.6$ | **0.42** |
| **Boolean** | (0.7 > 0.5) AND (0.6 > 0.5) | **True** |

The product t-norm *multiplies* satisfaction values, so even when all predicates are above 0.5, the conjunction can drop below 0.5. With three predicates each at 0.6: $0.6 \times 0.6 \times 0.6 = 0.216$.

For our expression `P(X) & (Q(X) | ~R(X))`, if P=0.6, Q=0.55, R=0.45, we have:
- **Soft**: $0.6 \times (0.55 + 0.55 - 0.55\times0.55) = 0.6 \times 0.80 = 0.48$
- **Boolean**: `True AND (True OR True) = True`

The two kinds of satisfaction ask different questions, and as a result we get different answers: 

Soft satisfaction measures the *degree* of truth, which is useful for computing gradients during training. The question it answers is "How well is the constraint satisfied?"

Boolean satisfaction** measures *hard* decisions, useful for evaluation metrics. The question it answers is "Is the constraint satisfied according to model predictions?"

This may be tricky, but the good news, however, is that when soft satisfaction increases, so will boolean satisfaction. This means that a natural way to train neural networks is to try to define a loss such that minimizing it maximizes soft satisfaction. 

## From logic to loss

We could use the logical expressions to automaticaly construct differentiable loss functions. We have a helper `logic_to_loss` to help with that. The interface of `logic_to_loss` is similar to that of `compile_logic` in that it takes the logical expression and the mapping from predicates to neurons, and generates a compiled expression that can be applied to a batch of examples.

In [45]:
loss_calculator = psn.logic_to_loss(expr, predicates)

Given a batch of examples `x` (we will re-use the batch we created above), we can now compute the value of loss. The loss can be interpreted as "how poorly does the current batch of examples satisfy the logical constraint". This means that minimizing the loss will lead to better satisfaction.

In [46]:
loss = loss_calculator.loss(X=x)
print(f"Loss = {loss}")

Loss = 33.08138656616211


The loss and the compiled expression underneath the hood are all differentiable. This means that we can use the standard torch machinery like doing a backward pass, and using any optimizer we want to improve the objective.

In [47]:
loss.backward()
print(f"\nGradients computed: {model_p[0].weight.grad is not None}")


Gradients computed: True


## Implication constraints

The library supports all standard logical operators. One particularly useful one in neural-symbolic learning is *implication*. The constraint $\forall X, P(X) \Rightarrow Q(X)$ reads "for every input $X$, if $P(X)$ holds, then $Q(X)$ must also hold".

This is a natural way to encode soft rules. For instance, if a model predicts that a sample belongs to class A, it may be constrained to also output high confidence for some related property B. Unlike hard boolean implication, the differentiable version allows gradients to flow through so the model can learn to satisfy the rule.

In [48]:
impl_expr = psn.Implies(P(X), Q(X))
print(f"Expression: {impl_expr}")
x = torch.rand(batch_size, 10)

impl_loss = psn.logic_to_loss(impl_expr, predicates)
impl_sat = impl_loss.satisfaction(X=x)
print(f"Satisfaction of P(X) -> Q(X): {impl_sat.item():.4f}")

Expression: Implies(P(X), Q(X))
Satisfaction of P(X) -> Q(X): 1.0000


## Multiple variables

So far, all our predicates have shared the same variable `X`. The library also supports separate variables bound to different input tensors. This is useful when different predicates operate on different data sources or modalities.

The constraint $\forall X, \forall Y, P(X) \wedge Q(Y)$ says "for all inputs $X$ and $Y$, both $P(X)$ and $Q(Y)$ should hold". When evaluating, we bind each variable to its own batch tensor using keyword arguments.

In [49]:
Y = psn.Variable("Y")

multi_expr = psn.And(P(X), Q(Y))
print(f"Expression: {multi_expr}")

multi_loss = psn.logic_to_loss(multi_expr, predicates)

x = torch.rand(batch_size, 10)
y = torch.rand(batch_size, 10)
multi_sat = multi_loss.satisfaction(X=x, Y=y)
print(f"Satisfaction of P(X) AND Q(Y): {multi_sat.item():.4f}")

Expression: P(X) & Q(Y)
Satisfaction of P(X) AND Q(Y): 0.0000


## Multi-class predicates and constant arguments

Predicates can accept *constant* arguments alongside variable ones. This is especially useful when a predicate is a multi-class classifier and we want to express that an input belongs to a specific class.

For example, suppose `Digit` is a 10-class classifier over handwritten digit images. The expression $\forall X, \text{Digit}(X, 5)$ reads "every input $X$ should be classified as digit 5". The constant `5` selects the appropriate output neuron. That is, `pysignet` automatically indexes into the correct column of the classifier's output.

The classifier returns a 10-dimensional softmax vector per sample. When called with a constant index, only the column at that index is extracted and used as the predicate's satisfaction value.

In [50]:
Digit = psn.Symbol("Digit")

class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 10)

    def forward(self, x):
        return torch.softmax(self.fc(x), dim=-1)

digit_model = DigitClassifier()

# Digit(X, 5) means "input X is classified as digit 5"
digit_expr = Digit(X, 5)
print(f"Expression: {digit_expr}")
print(f"Interpretation: All inputs should be classified as digit 5")

digit_loss = psn.logic_to_loss(digit_expr, {"Digit": digit_model})

x_digits = torch.rand(batch_size, 784)
digit_sat = digit_loss.satisfaction(X=x_digits)
print(f"Satisfaction (randomly initialised classifier): {digit_sat.item():.4f}")

Expression: Digit(X, 5)
Interpretation: All inputs should be classified as digit 5
Satisfaction (randomly initialised classifier): 0.0000


## A training loop

Now that we have seen all the components, let us put them together in a simple training loop. We will train `model_p` to satisfy the implication constraint $\forall X, P(X) \Rightarrow Q(X)$ over random batches.

Since `pred_q` is a deterministic function (not a learnable model), `model_p` must adapt its outputs so that whenever `P(X)` is high, `Q(X)` is also high. The loss decreases as the implication becomes better satisfied.

In [52]:
training_loss = psn.logic_to_loss(impl_expr, predicates)
optimizer = torch.optim.Adam(model_p.parameters(), lr=0.01)
for epoch in range(10):
    x_batch = torch.rand(2, 10)

    loss = training_loss.loss(X=x_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        sat = training_loss.satisfaction(X=x_batch)

    print(f"Epoch {epoch + 1:2d}: loss = {loss.item():.4f}, satisfaction = {sat.item():.4f}")

Epoch  1: loss = -0.0000, satisfaction = 1.0000
Epoch  2: loss = -0.0000, satisfaction = 1.0000
Epoch  3: loss = -0.0000, satisfaction = 1.0000
Epoch  4: loss = -0.0000, satisfaction = 1.0000
Epoch  5: loss = -0.0000, satisfaction = 1.0000
Epoch  6: loss = -0.0000, satisfaction = 1.0000
Epoch  7: loss = -0.0000, satisfaction = 1.0000
Epoch  8: loss = -0.0000, satisfaction = 1.0000
Epoch  9: loss = -0.0000, satisfaction = 1.0000
Epoch 10: loss = -0.0000, satisfaction = 1.0000


The patterns shown here form the core of `pysignet`:

- Combine `And`, `Or`, `Not`, `Implies`, and `Equivalent` freely to express any propositional constraint.
- Use multiple `Variable`s bound to different data sources for cross-modal or relational constraints.
- Pass integer constants as predicate arguments to select specific output neurons of multi-class classifiers.
- Use `compile_logic` when you need per-sample satisfaction values, and `logic_to_loss` when you need a scalar loss for training.

A natural next step is to see [the notebook that shows how we can build a standard MNIST classifier with this library](MNIST.ipynb).